# 1. Base Setup

## 1.1 Install packages

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost
!pip install lightgbm

## 1.2 Load all needed imports

In [21]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix, accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [22]:
import cf_copilot
print(cf_copilot.__file__)

/home/jeroenewalts/code/EwaltsJ/cf_copilot/cf_copilot/__init__.py


In [23]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [24]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)

# Example: train+valid for search, final holdout for test
valid_cutoff = big_df["reference_date"].quantile(0.6)
test_cutoff = big_df["reference_date"].quantile(0.8)

search_df = big_df[big_df["reference_date"] <= test_cutoff].copy()
final_test_df = big_df[big_df["reference_date"] > test_cutoff].copy()

test_fold = np.where(search_df["reference_date"] <= valid_cutoff, -1, 0)
ps = PredefinedSplit(test_fold)

X_search, y_search = preprocess(search_df)
X_final_test, y_final_test = preprocess(final_test_df)

Loading local file: /home/jeroenewalts/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [25]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [26]:
from lightgbm import LGBMClassifier

classifier = LGBMClassifier(
    objective="multiclass",
    random_state=42,
    n_jobs=4,
    verbose=-1,
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])

In [27]:
pipeline.get_params()

{'memory': None,
 'steps': [('preprocessor',
   ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                    ['business_year', 'invoice_age_days',
                                     'days_until_due', 'pay_terms_days',
                                     'invoice_month', 'due_month', 'days_past_due',
                                     'customer_avg_delay', 'late_payment_ratio',
                                     'prev_transaction_count',
                                     'days_since_last_invoice',
                                     'customer_risk_score', 'invoice_amount',
                                     'invoice_amount_log']),
                                   ('cat',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(fill_value=-1,
                                                                   strategy='constant')),
                     

In [28]:
import datetime
print("START:", datetime.datetime.now())

param_distributions = {
    # Core boosting capacity
    "classifier__n_estimators": randint(300, 1001),
    "classifier__learning_rate": uniform(0.02, 0.10),
    "classifier__num_leaves": randint(15, 64),
    "classifier__max_depth": [4, 6, 8, 10],

    # Split / leaf control
    "classifier__min_child_samples": randint(20, 81),
    "classifier__min_child_weight": uniform(1e-3, 0.2),
    "classifier__min_split_gain": uniform(0.0, 0.2),

    # Sampling
    "classifier__subsample": uniform(0.7, 0.3),
    "classifier__subsample_freq": randint(1, 5),
    "classifier__colsample_bytree": uniform(0.7, 0.3),

    # Regularization
    "classifier__reg_alpha": uniform(0.0, 1.0),
    "classifier__reg_lambda": uniform(0.0, 1.0),
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=60,
    cv=ps,
    scoring="neg_log_loss",
    n_jobs=4,
    verbose=2,
    random_state=42,
    error_score=np.nan,
    return_train_score=True,
)

random_search.fit(X_search, y_search)

best_model = random_search.best_estimator_

START: 2026-03-24 22:45:00.611025
Fitting 1 folds for each of 60 candidates, totalling 60 fits


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__colsample_bytree=0.9040922615763338, classifier__learning_rate=0.06504992519695431, classifier__max_depth=6, classifier__min_child_samples=23, classifier__min_child_weight=0.18944035113697055, classifier__min_split_gain=0.11265764356910786, classifier__n_estimators=645, classifier__num_leaves=16, classifier__reg_alpha=0.6842330265121569, classifier__reg_lambda=0.4401524937396013, classifier__subsample=0.7366114704534336, classifier__subsample_freq=1; total time=   6.8s


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__colsample_bytree=0.8952665418846558, classifier__learning_rate=0.025641157902710026, classifier__max_depth=10, classifier__min_child_samples=63, classifier__min_child_weight=0.18871054180315006, classifier__min_split_gain=0.00015575316820286568, classifier__n_estimators=576, classifier__num_leaves=47, classifier__reg_alpha=0.3042422429595377, classifier__reg_lambda=0.5247564316322378, classifier__subsample=0.8295835055926347, classifier__subsample_freq=1; total time=  10.6s


...

[CV] END classifier__colsample_bytree=0.8378040423838516, classifier__learning_rate=0.10420914150187098, classifier__max_depth=4, classifier__min_child_samples=79, classifier__min_child_weight=0.014247195570373173, classifier__min_split_gain=0.009172253280926591, classifier__n_estimators=352, classifier__num_leaves=16, classifier__reg_alpha=0.3287516102875082, classifier__reg_lambda=0.6334008543167258, classifier__subsample=0.7720436856334578, classifier__subsample_freq=3; total time=   4.0s


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__colsample_bytree=0.861179025014221, classifier__learning_rate=0.06601191589835721, classifier__max_depth=10, classifier__min_child_samples=50, classifier__min_child_weight=0.08106009778205628, classifier__min_split_gain=0.13953351473938636, classifier__n_estimators=659, classifier__num_leaves=18, classifier__reg_alpha=0.3456672833238632, classifier__reg_lambda=0.8967884099060118, classifier__subsample=0.8421884920788616, classifier__subsample_freq=1; total time=   7.3s
[CV] END classifier__colsample_bytree=0.8176732135299196, classifier__learning_rate=0.06374749220237291, classifier__max_depth=8, classifier__min_child_samples=53, classifier__min_child_weight=0.07065109340466007, classifier__min_split_gain=0.10279789783196217, classifier__n_estimators=634, classifier__num_leaves=44, classifier__reg_alpha=0.6220867002278735, classifier__reg_lambda=0.8623637087467452, classifier__subsample=0.9848561870972925, classifier__subsample_freq=4; total time=  10.3s


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[CV] END classifier__colsample_bytree=0.9919331047810312, classifier__learning_rate=0.08019353955409603, classifier__max_depth=8, classifier__min_child_samples=38, classifier__min_child_weight=0.16535812736888986, classifier__min_split_gain=0.06901652556447735, classifier__n_estimators=981, classifier__num_leaves=36, classifier__reg_alpha=0.08870253375705561, classifier__reg_lambda=0.12063587110060081, classifier__subsample=0.8382336304098177, classifier__subsample_freq=2; total time=  14.0s


In [29]:
y_proba = best_model.predict_proba(X_final_test)
y_pred = best_model.predict(X_final_test)

clf_classes = np.array(best_model.named_steps["classifier"].classes_)

search_classes = np.array(sorted(pd.Series(y_search).unique()))
test_classes = np.array(sorted(pd.Series(y_final_test).unique()))

cv_results = pd.DataFrame(random_search.cv_results_)
n_failed = cv_results["mean_test_score"].isna().sum()
print(f"Failed parameter sets: {n_failed} / {len(cv_results)}")

missing_in_train = set(test_classes) - set(clf_classes)
print("Classes in final test but not seen during search:", missing_in_train)

if missing_in_train:
    raise ValueError(
        f"Final test contains unseen classes: {missing_in_train}. "
        "The model cannot evaluate those correctly."
    )

if y_proba.shape[1] != len(clf_classes):
    raise ValueError(
        f"Mismatch between probability columns ({y_proba.shape[1]}) "
        f"and classifier classes ({len(clf_classes)})."
    )

print("Best CV log_loss:", -random_search.best_score_)
print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

final_log_loss = log_loss(y_final_test, y_proba, labels=clf_classes)
final_accuracy = accuracy_score(y_final_test, y_pred)
final_f1_macro = f1_score(y_final_test, y_pred, average="macro")

print("\nFinal holdout performance")
print(f"log_loss  : {final_log_loss:.6f}")
print(f"accuracy  : {final_accuracy:.6f}")
print(f"f1_macro  : {final_f1_macro:.6f}")

print("\nClass checks")
print("Search classes:", list(search_classes))
print("Test classes  :", list(test_classes))
print("Model classes :", list(clf_classes))

best_idx = random_search.best_index_
best_train_score = random_search.cv_results_["mean_train_score"][best_idx]
best_valid_score = random_search.cv_results_["mean_test_score"][best_idx]

print("\nBest CV split scores")
print(f"Best mean train neg_log_loss: {best_train_score:.6f}")
print(f"Best mean valid neg_log_loss: {best_valid_score:.6f}")
print(f"Best mean train log_loss    : {-best_train_score:.6f}")
print(f"Best mean valid log_loss    : {-best_valid_score:.6f}")

display(
    cv_results.sort_values("rank_test_score")[
        ["rank_test_score", "mean_test_score", "mean_train_score", "params"]
    ].head(5)
)

/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Failed parameter sets: 0 / 60
Classes in final test but not seen during search: set()
Best CV log_loss: 0.7987655454754763
Best params:
  classifier__colsample_bytree: 0.7992694074557947
  classifier__learning_rate: 0.026355835028602365
  classifier__max_depth: 8
  classifier__min_child_samples: 43
  classifier__min_child_weight: 0.13418447132349934
  classifier__min_split_gain: 0.11825955754154543
  classifier__n_estimators: 846
  classifier__num_leaves: 15
  classifier__reg_alpha: 0.38292687475378984
  classifier__reg_lambda: 0.9717120953891037
  classifier__subsample: 0.954674147279825
  classifier__subsample_freq: 1

Final holdout performance
log_loss  : 0.719742
accuracy  : 0.751843
f1_macro  : 0.558085

Class checks
Search classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Test classes  : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Model classes : [np.int64(1), np.int64(2), np.int64(

,rank_test_score,mean_test_score,mean_train_score,params
8,1,-0.798766,-0.508178,{'classifier__colsample_bytree': 0.79926940745...
28,2,-0.799387,-0.531633,{'classifier__colsample_bytree': 0.96598514468...
35,3,-0.801060,-0.399383,{'classifier__colsample_bytree': 0.89760808948...
46,4,-0.801994,-0.574166,{'classifier__colsample_bytree': 0.90431182812...
6,5,-0.802299,-0.415242,{'classifier__colsample_bytree': 0.80702599800...


In [30]:
from cf_copilot.cashflow_prediction.evaluation import build_actual_weekly_cf, compare_forecast_vs_actual, compute_forecast_metrics
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES
from colorama import Fore, Style
from cf_copilot.ml_logic.encoders import preprocess


def evaluate_forecast_holdout(
    model: Pipeline,
    test_df: pd.DataFrame,
    verbose: bool = True
) -> tuple[dict, dict]:
    """
    Evaluate forecast quality on the holdout set by reference_date.

    Args:
        model: fitted sklearn Pipeline
        test_df: holdout snapshot dataframe
        verbose: whether to print evaluation output

    Returns:
        forecast_metrics: scalar metrics to merge into MLflow metrics
        forecast_summary: JSON-serializable artifact with per-reference-date detail
    """
    if verbose:
        print(Fore.BLUE + f"\nEvaluating forecast on {len(test_df)} rows..." + Style.RESET_ALL)

    if model is None:
        if verbose:
            print("❌ No model to evaluate forecast")
        return {}, {}

    if len(test_df) == 0:
        if verbose:
            print("❌ No holdout rows available for forecast evaluation")
        return {}, {}

    per_reference_results = []
    if hasattr(model, "named_steps") and "classifier" in model.named_steps:
        model_classes = model.named_steps["classifier"].classes_
    else:
        model_classes = model.classes_

    class_to_index = {int(c): i for i, c in enumerate(model_classes)}

    for reference_date, snapshot_df in test_df.groupby("reference_date"):
        snapshot_df = snapshot_df.copy()
        if len(snapshot_df) == 0:
            continue

        X_snapshot, _ = preprocess(snapshot_df)
        probas = model.predict_proba(X_snapshot)

        pred_cash_df = snapshot_df.copy()

        for w in WEEK_CLASSES:
            pred_cash_df[f"p_{w}"] = (
                probas[:, class_to_index[int(w)]]
                if int(w) in class_to_index
                else 0.0
            )

        weekly_forecast_df = pd.DataFrame([
            {
                "week_bucket": int(w),
                "forecast_cash": round(
                    float((pred_cash_df["total_open_amount"] * pred_cash_df[f"p_{w}"]).sum()),
                    2,
                ),
            }
            for w in WEEK_CLASSES
        ])

        total_invoice_amount = round(float(pred_cash_df["total_open_amount"].sum()), 2)
        total_expected_cash = round(float(weekly_forecast_df["forecast_cash"].sum()), 2)

        if total_expected_cash - total_invoice_amount > 0:
            print("Expected cash exceeds total invoice amount. This should not happen.")

        actual_weekly_df = build_actual_weekly_cf(
            invoices_df=snapshot_df,
            reference_date=pd.Timestamp(reference_date),
        )

        comparison_df = compare_forecast_vs_actual(
            weekly_forecast_df=weekly_forecast_df,
            actual_weekly_df=actual_weekly_df,
        )

        snapshot_metrics = compute_forecast_metrics(comparison_df)

        per_reference_results.append({
            "reference_date": str(pd.Timestamp(reference_date).date()),
            "forecast_check": {
                "total_invoice_amount": total_invoice_amount,
                "total_expected_cash": total_expected_cash,
            },
            "forecast_metrics": {
                "forecast_mae_weekly": float(snapshot_metrics["MAE (weekly)"]),
                "forecast_mape_weekly": (
                    float(snapshot_metrics["MAPE (weekly)"])
                    if pd.notna(snapshot_metrics["MAPE (weekly)"])
                    else None
                ),
                "total_actual_cf": float(snapshot_metrics["Total actual cf"]),
                "total_forecast_cf": float(snapshot_metrics["Total forecast cf"]),
                "total_cf_difference": float(snapshot_metrics["Total cf difference"]),
            },
        })

    if not per_reference_results:
        if verbose:
            print("❌ No per-reference-date forecast results available")
        return {}, {}

    mae_values = [r["forecast_metrics"]["forecast_mae_weekly"] for r in per_reference_results]
    mape_values = [
        r["forecast_metrics"]["forecast_mape_weekly"]
        for r in per_reference_results
        if r["forecast_metrics"]["forecast_mape_weekly"] is not None
    ]
    total_actual_values = [r["forecast_metrics"]["total_actual_cf"] for r in per_reference_results]
    total_forecast_values = [r["forecast_metrics"]["total_forecast_cf"] for r in per_reference_results]
    total_diff_values = [r["forecast_metrics"]["total_cf_difference"] for r in per_reference_results]

    forecast_metrics = {
        "forecast_mae_weekly": float(np.mean(mae_values)),
        "forecast_mape_weekly": float(np.mean(mape_values)) if mape_values else np.nan,
        "forecast_total_actual_cf": float(np.mean(total_actual_values)),
        "forecast_total_forecast_cf": float(np.mean(total_forecast_values)),
        "forecast_total_cf_difference": float(np.mean(total_diff_values)),
    }

    forecast_summary = {
        "per_reference_date": per_reference_results,
        "aggregate": {
            "forecast_mae_weekly": forecast_metrics["forecast_mae_weekly"],
            "forecast_mape_weekly": (
                forecast_metrics["forecast_mape_weekly"]
                if pd.notna(forecast_metrics["forecast_mape_weekly"])
                else None
            ),
            "forecast_total_actual_cf": forecast_metrics["forecast_total_actual_cf"],
            "forecast_total_forecast_cf": forecast_metrics["forecast_total_forecast_cf"],
            "forecast_total_cf_difference": forecast_metrics["forecast_total_cf_difference"]
        },
    }

    if verbose:
        print(f"✅ Forecast MAE weekly: {forecast_metrics['forecast_mae_weekly']:.2f}")

        if pd.notna(forecast_metrics["forecast_mape_weekly"]):
            print(f"✅ Forecast MAPE weekly: {forecast_metrics['forecast_mape_weekly']:.4f}")
        else:
            print("✅ Forecast MAPE weekly: None")

        print(f"✅ Forecast total actual cf: {forecast_metrics['forecast_total_actual_cf']:.2f}")
        print(f"✅ Forecast total forecast cf: {forecast_metrics['forecast_total_forecast_cf']:.2f}")
        print(f"✅ Forecast total cf difference: {forecast_metrics['forecast_total_cf_difference']:.2f}")

    return forecast_metrics, forecast_summary

In [31]:
forecast_metrics, forecast_summary = evaluate_forecast_holdout(
    model=best_model,
    test_df=final_test_df,
    verbose=True,
)

print("\nAggregate forecast metrics")
for k, v in forecast_metrics.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics])

print("END:", datetime.datetime.now())


Evaluating forecast on 18444 rows...


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.

✅ Forecast MAE weekly: 557370.00
✅ Forecast MAPE weekly: 0.3370
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27833929.31
✅ Forecast total cf difference: -72188.82

Aggregate forecast metrics
forecast_mae_weekly: 557369.9990000001
forecast_mape_weekly: 0.3370000000000001
forecast_total_actual_cf: 27906118.129499994
forecast_total_forecast_cf: 27833929.310000002
forecast_total_cf_difference: -72188.81950000001
END: 2026-03-24 22:49:17.555184


/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jeroenewalts/.pyenv/versions/cf_copilot/lib/python3.10/site-packages/sklearn/utils/validation.